In [1]:
# Domain Model
#
# Question:
# Among visitors who took at least one test,
# how does the DOMAIN of the first test relate to signup probability,
# controlling for device and geography?
#
# Population:
# Visitors with ≥1 test taken
#
# Outcome:
# has_signup (binary)

In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
import statsmodels.api as sm

In [3]:
DATA_PATH = "../data/visitor_features_engineered.parquet"

df = pd.read_parquet(DATA_PATH)
df.shape

(535391, 15)

In [5]:
print(df['first_test_domain'].value_counts(dropna=False))

first_test_domain
Mood & Depression                  109218
Personality Disorders & Traits     106368
Trauma & Dissociation              105351
Sexual & Gender Health              62306
Addiction & Compulsive Behavior     56593
No Test                             32656
Anxiety & Stress                    30685
Life, Work & Physical Health        21660
Neurodevelopmental & Cognitive      10554
Name: count, dtype: int64


In [6]:
df_domain = df[df["first_test_domain"] != "No Test"].copy()

df_domain.shape

(502735, 15)

In [7]:
TARGET = "has_signup"

CATEGORICAL_FEATURES = [
    "browser",
    "os_name_clean",
    "sub_region",
    "first_test_domain",
]

df_model = df_domain[CATEGORICAL_FEATURES + [TARGET]].copy()
df_model[TARGET] = df_model[TARGET].astype(int)

df_model.head()

,browser,os_name_clean,sub_region,first_test_domain,has_signup
2,mobile_web,iOS,East South Central,Trauma & Dissociation,0
3,mobile_web,iOS,East South Central,Sexual & Gender Health,0
4,mobile_web,Android,East South Central,Anxiety & Stress,0
5,mobile_web,iOS,East South Central,Neurodevelopmental & Cognitive,0
7,mobile_web,Android,East South Central,Personality Disorders & Traits,0


In [8]:
df_model["first_test_domain"].value_counts()

first_test_domain
Mood & Depression                  109218
Personality Disorders & Traits     106368
Trauma & Dissociation              105351
Sexual & Gender Health              62306
Addiction & Compulsive Behavior     56593
Anxiety & Stress                    30685
Life, Work & Physical Health        21660
Neurodevelopmental & Cognitive      10554
Name: count, dtype: int64

In [9]:
REFERENCE_CATEGORIES = {
    "browser": "mobile_web",
    "os_name_clean": "iOS",
    "sub_region": "South Atlantic",
    "first_test_domain": "Mood & Depression",
}

In [10]:
encoder = OneHotEncoder(
    drop=None,
    sparse_output=False,
    handle_unknown="ignore"
)

encoder.set_output(transform="pandas")

X_encoded = encoder.fit_transform(df_model[CATEGORICAL_FEATURES])

In [11]:
X_encoded.head()

,browser_desktop_web,browser_mobile_web,os_name_clean_Android,os_name_clean_Chrome OS,os_name_clean_Other,os_name_clean_Windows,os_name_clean_iOS,os_name_clean_macOS,sub_region_East North Central,sub_region_East South Central,...,sub_region_West North Central,sub_region_West South Central,first_test_domain_Addiction & Compulsive Behavior,first_test_domain_Anxiety & Stress,"first_test_domain_Life, Work & Physical Health",first_test_domain_Mood & Depression,first_test_domain_Neurodevelopmental & Cognitive,first_test_domain_Personality Disorders & Traits,first_test_domain_Sexual & Gender Health,first_test_domain_Trauma & Dissociation
2,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0
5,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
7,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [12]:
for col, ref in REFERENCE_CATEGORIES.items():
    ref_col = f"{col}_{ref}"
    if ref_col in X_encoded.columns:
        X_encoded = X_encoded.drop(columns=ref_col)

In [14]:
X_encoded.head()

,browser_desktop_web,os_name_clean_Android,os_name_clean_Chrome OS,os_name_clean_Other,os_name_clean_Windows,os_name_clean_macOS,sub_region_East North Central,sub_region_East South Central,sub_region_Middle Atlantic,sub_region_Mountain,...,sub_region_Pacific,sub_region_West North Central,sub_region_West South Central,first_test_domain_Addiction & Compulsive Behavior,first_test_domain_Anxiety & Stress,"first_test_domain_Life, Work & Physical Health",first_test_domain_Neurodevelopmental & Cognitive,first_test_domain_Personality Disorders & Traits,first_test_domain_Sexual & Gender Health,first_test_domain_Trauma & Dissociation
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
7,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [13]:
X = sm.add_constant(X_encoded)
y = df_model[TARGET]

X.shape, y.mean()

((502735, 22), np.float64(0.017263568281500195))

In [15]:
logit = sm.Logit(y, X)
result = logit.fit(disp=False)

result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                           Logit Regression Results                           
==============================================================================
Dep. Variable:             has_signup   No. Observations:               502735
Model:                          Logit   Df Residuals:                   502713
Method:                           MLE   Df Model:                           21
Date:                Fri, 16 Jan 2026   Pseudo R-squ.:                 0.01350
Time:                        12:54:16   Log-Likelihood:                -43241.
converged:                       True   LL-Null:                       -43833.
Covariance Type:            nonrobust   LLR p-value:                2.168e-237
=====================================================================================================================
                                                        coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------------------------------------
const                                                -3.6655      0.029   -124.523      0.000      -3.723      -3.608
browser_desktop_web                                  -1.4133      0.098    -14.431      0.000      -1.605      -1.221
os_name_clean_Android                                 0.2560      0.024     10.673      0.000       0.209       0.303
os_name_clean_Chrome OS                               1.6641      0.130     12.822      0.000       1.410       1.918
os_name_clean_Other                                   0.0417      0.198      0.210      0.834      -0.347       0.430
os_name_clean_Windows                                 1.0761      0.112      9.635      0.000       0.857       1.295
os_name_clean_macOS                                   0.9859      0.129      7.635      0.000       0.733       1.239
sub_region_East North Central                        -0.0268      0.036     -0.739      0.460      -0.098       0.044
sub_region_East South Central                         0.2273      0.043      5.300      0.000       0.143       0.311
sub_region_Middle Atlantic                           -0.0711      0.040     -1.770      0.077      -0.150       0.008
sub_region_Mountain                                  -0.1748      0.047     -3.686      0.000      -0.268      -0.082
sub_region_New England                               -0.2183      0.066     -3.310      0.001      -0.348      -0.089
sub_region_Pacific                                   -0.1725      0.039     -4.369      0.000      -0.250      -0.095
sub_region_West North Central                        -0.0123      0.048     -0.257      0.797      -0.106       0.082
sub_region_West South Central                         0.1181      0.037      3.197      0.001       0.046       0.191
first_test_domain_Addiction & Compulsive Behavior    -0.5428      0.040    -13.457      0.000      -0.622      -0.464
first_test_domain_Anxiety & Stress                   -0.5588      0.054    -10.327      0.000      -0.665      -0.453
first_test_domain_Life, Work & Physical Health       -0.4439      0.057     -7.827      0.000      -0.555      -0.333
first_test_domain_Neurodevelopmental & Cognitive     -0.7482      0.107     -6.970      0.000      -0.959      -0.538
first_test_domain_Personality Disorders & Traits     -0.4511      0.032    -14.199      0.000      -0.513      -0.389
first_test_domain_Sexual & Gender Health             -0.5053      0.038    -13.187      0.000      -0.580      -0.430
first_test_domain_Trauma & Dissociation              -0.4570      0.032    -14.406      0.000      -0.519      -0.395
=====================================================================================================================
"""

In [16]:
odds_ratios = pd.DataFrame({
    "odds_ratio": np.exp(result.params),
    "ci_lower": np.exp(result.conf_int()[0]),
    "ci_upper": np.exp(result.conf_int()[1]),
    "p_value": result.pvalues
})

odds_ratios.sort_values("odds_ratio", ascending=False)

,odds_ratio,ci_lower,ci_upper,p_value
os_name_clean_Chrome OS,5.280729,4.094675,6.810333,1.241920e-37
os_name_clean_Windows,2.933257,2.356548,3.651102,5.716446e-22
os_name_clean_macOS,2.680167,2.080918,3.451984,2.251477e-14
os_name_clean_Android,1.291794,1.232463,1.353981,1.361269e-26
sub_region_East South Central,1.255170,1.153994,1.365215,1.156288e-07
sub_region_West South Central,1.125394,1.046772,1.209922,1.388519e-03
os_name_clean_Other,1.042553,0.706757,1.537893,8.335855e-01
sub_region_West North Central,0.987737,0.899139,1.085066,7.969276e-01
sub_region_East North Central,0.973538,0.906659,1.045349,4.601677e-01
sub_region_Middle Atlantic,0.931415,0.860948,1.007650,7.670824e-02


In [17]:
# Interpretation:
#
# Odds ratios compare each domain to:
#   - Mood & Depression
#   - desktop_web
#   - iOS
#   - South Atlantic
#
# This isolates "what kind of help" the visitor sought,
# not "how engaged" they were.